# 21 - Health check before documentation (Part A: calibration, the threshold and the crop rule)

Questions put to the deployed system before the report is written, so that a problem is found
now and not after the chapters. **Part A** uses only the scores notebooks 15 and 17 saved: no
GPU, about 10 minutes. Part B (blurred photos and the phone's photo size) and Part C (a new
field test set, scored once) follow in their own sections.

**What is deployed:** the ensemble of v2 ResNet-50 seed 44, ConvNeXt-Tiny at 1e-4 and
EfficientNet-B0; temperature 2.4381, fitted on the selection set in notebook 18b; referral
threshold 0.8, a policy chosen after notebook 18b's threshold table (wrong diagnoses judged
worse than referrals) and reported as such. The confidence shown is the 11-class probability
of the best class of the farmer's crop, **not rescaled** over that crop's classes.

**Rules, written on 31 Oct 2026 before this notebook was first run.** Only A3 can change
anything deployed; every other question is reported only.

- **A1 Calibration, before and after temperature scaling**, on every set: validation (the set
  NFR i names), test, the field photos of each, the selection set (out of fold, since the
  temperature was fitted on it) and PlantDoc. Measured both as the app shows it (crop given)
  and over all 11 classes. Report only.
- **A2 What each threshold does**, on every set: share answered, accuracy of the answered,
  wrong answers per 100 photos and share sent to an officer. 0.8 is deployed; 0.5 to 0.9 are
  shown for comparison only. Report only.
- **A3 Is the shown confidence too low because it is not rescaled within the crop?**
  Candidate: temperature scaling fitted on the farmer's crop's classes only (the other crops'
  scores removed), so the confidence is a probability over that crop's diseases. It never
  changes the diagnosis. The safety guard stays: a photo whose chosen crop holds less than
  half of the deployed 11-class probability is always referred, as now.
  **Adopted only if both hold on the selection set, out of fold:** (a) its ECE is lower, with
  the whole paired bootstrap range of the difference below zero; (b) it refers no worse: the
  paired bootstrap range of the AURC difference does not lie wholly above zero.
  **If adopted, the threshold is re-derived by the policy behind 0.8, not chosen again:** the
  cost of a wrong answer, in referrals, at which 0.8 is the lowest-cost threshold for the
  current confidences, applied unchanged to the new ones. If not adopted, nothing changes.
- **A4 Per class at the deployed threshold:** which diseases are answered, which never are,
  and where the wrong answers fall. Report only.
- **A5 Crop not given** (mixed batches): how often the crop is detected correctly, and the
  accuracy and answers that follow. Report only.

In [ ]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

In [ ]:
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

In [ ]:
%pip install -q -r {ML_DIR}/requirements.txt

## Settings and the saved scores

In [ ]:
import importlib
import json
import shutil

import numpy as np
import pandas as pd

from src import calibration as cal, metrics, training
for module in (cal, metrics, training):
    importlib.reload(module)

FOLDS, BOOTSTRAP, FLOOR = 5, 1000, 0.5
SEED = config.RANDOM_SEED
DEPLOYED_THRESHOLD = 0.8
THRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9]

N18B = ML_DIR / "results" / "ensemble_deployment" / "decision.json"
deployed = json.loads(N18B.read_text())["deployed"]
MEMBERS = deployed["model_runs"]
TEMPERATURE = deployed["temperature"]
assert MEMBERS == ["d17_resnet50_seed44", "d22_convnext_tiny_lr1e-04", "d22_efficientnet_b0"], MEMBERS
assert abs(TEMPERATURE - 2.4381) < 1e-3, TEMPERATURE
FOLDER = {"d17": config.MODELS_DIR / "d17_comparison", "d22": config.MODELS_DIR / "d22_comparison"}
OUT_DIR = config.MODELS_DIR / "health_check"
REPO_RESULTS = ML_DIR / "results" / "health_check"
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPO_RESULTS.mkdir(parents=True, exist_ok=True)

EVAL_SETS = ["val", "test", "ood_test", "selection"]
splits = {name: pd.read_csv(config.SPLITS_DIR / f"{name}.csv") for name in EVAL_SETS}
CLASSES = list(config.PLANTVILLAGE_CLASSES)
INDEX = {name: i for i, name in enumerate(CLASSES)}
CLASS_CROPS = np.array([{"Corn_(maize)": "maize", "Tomato": "tomato", "Potato": "potato"}[c.split("___")[0]]
                        for c in CLASSES])
CROPS = ["maize", "tomato", "potato"]
SHORT = {c: c.replace("Corn_(maize)___", "Maize ").replace("___", " ").replace("_", " ").strip() for c in CLASSES}
FIELD_SOURCES = ["fieldplant", "tanzania"]

PARTS = {}
for name in EVAL_SETS:
    labels = splits[name]["class"].map(INDEX)
    keep = labels.notna().to_numpy()
    lab = labels[keep].astype(int).to_numpy()
    PARTS[name] = {"rows": splits[name][keep].reset_index(drop=True), "keep": keep, "labels": lab, "crops": CLASS_CROPS[lab]}
FIELD = {name: PARTS[name]["rows"]["source"].isin(FIELD_SOURCES).to_numpy() for name in ["val", "test"]}


def member_scores(run, name):
    folder = FOLDER["d22"] if run.startswith("d22_") else FOLDER["d17"]
    return np.load(folder / f"{run}__{name}.npy")[PARTS[name]["keep"]]


# The ensemble's scores, exactly as the backend computes them: log of the members' averaged probabilities.
SCORES = {name: np.log(np.mean([metrics.softmax(member_scores(m, name)) for m in MEMBERS], axis=0) + 1e-12)
          for name in EVAL_SETS}
SETS = [("validation", "val", None), ("validation: field photos", "val", FIELD["val"]),
        ("test", "test", None), ("test: field photos", "test", FIELD["test"]),
        ("selection (new field sources, out of fold)", "selection", None), ("PlantDoc (web photos)", "ood_test", None)]
print("Ensemble:", MEMBERS, "| T =", round(TEMPERATURE, 4), "| photos:", {n: len(PARTS[n]["labels"]) for n in EVAL_SETS})

In [ ]:
DEPLOYED = cal.Calibrator("temperature", {"temperature": TEMPERATURE})
SELECTION = PARTS["selection"]
# The temperature was fitted on the selection set, so its photos are scored out of fold, as in notebook 18b.
OOF = cal.out_of_fold("temperature", SCORES["selection"], SELECTION["labels"], FOLDS, SEED)


def part(name, mask=None):
    """Scores, true labels and the farmer's crop of one set (or the photos in mask)."""
    p = PARTS[name]
    if mask is None:
        return SCORES[name], p["labels"], p["crops"]
    return SCORES[name][mask], p["labels"][mask], p["crops"][mask]


def calibrated(name, mask=None):
    """The app's probabilities: the deployed temperature; the selection set out of fold."""
    if name == "selection":
        return OOF if mask is None else OOF[mask]
    return DEPLOYED.probabilities(part(name, mask)[0])


def shown(probs, labels, crops):
    """The confidence the app shows (best class of the farmer's crop) and whether that class is right."""
    conf, predicted = cal.crop_confidence(probs, CLASS_CROPS, crops)
    return conf, predicted == labels


def ece_range(conf, ok):
    return cal.bootstrap(metrics.expected_calibration_error, conf, ok, n=BOOTSTRAP, seed=SEED)

## A1 - Calibration before and after temperature scaling (report only)

*Gap* is mean confidence minus accuracy: negative means the app says it is less sure than it
really is (under-confident), positive means more sure (over-confident).

In [ ]:
a1_rows = []
for label, name, mask in SETS:
    lg, labels, crops = part(name, mask)
    for view in ["crop given (as the app shows it)", "all 11 classes"]:
        row = {"set": label, "view": view, "photos": len(labels)}
        for stage, probs in [("before", metrics.softmax(lg)), ("after", calibrated(name, mask))]:
            if view.startswith("crop"):
                conf, ok = shown(probs, labels, crops)
            else:
                conf, ok = probs.max(axis=1), probs.argmax(axis=1) == labels
            low, high = ece_range(conf, ok)
            row.update({"accuracy": ok.mean(), f"mean confidence {stage}": conf.mean(),
                        f"gap {stage}": conf.mean() - ok.mean(),
                        f"ECE {stage}": metrics.expected_calibration_error(conf, ok),
                        f"ECE {stage} low": low, f"ECE {stage} high": high})
        row["ECE lower after"] = row["ECE after"] < row["ECE before"]
        a1_rows.append(row)
a1 = pd.DataFrame(a1_rows).set_index(["set", "view"])
display(a1.round(4))

## A2 - What each threshold does, on every set (report only)

0.8 is deployed. *Wrong per 100* counts photos answered with the wrong disease, out of every
100 photos sent; *to officer* is the share referred.

In [ ]:
a2_rows = []
for label, name, mask in SETS:
    _, labels, crops = part(name, mask)
    conf, ok = shown(calibrated(name, mask), labels, crops)
    for t in THRESHOLDS:
        at = cal.at_threshold(conf, ok, t)
        a2_rows.append({"set": label, "threshold": t, "photos": len(ok), "accuracy, all photos": ok.mean(),
                        "answered": at["coverage"], "accuracy of answered": at["selective_accuracy"],
                        "wrong per 100": 100 * at["wrong_and_answered"], "to officer": at["referred"]})
a2 = pd.DataFrame(a2_rows).set_index(["set", "threshold"])
display(a2.round(3))

## A3 - Rescaling the confidence within the farmer's crop (rule fixed above)

The candidate removes the other crops' scores before temperature scaling, so the confidence
becomes a probability over the chosen crop's diseases. The diagnosis cannot change: the best
class of the crop is the same either way (checked below).

In [ ]:
NEG = -1e4  # far below any real score (log-probabilities here are above -28)


def within_crop(lg, crops):
    """Scores of the farmer's crop's classes only; the other crops' classes can never be chosen."""
    keep = CLASS_CROPS[None, :] == np.asarray(crops)[:, None]
    return np.where(keep, lg, NEG)


def crop_share(name, mask=None):
    """Share of the deployed 11-class probability on the farmer's crop. Below 0.5 the photo is
    always referred (the existing safety guard), whatever the rescaled confidence says."""
    _, _, crops = part(name, mask)
    keep = CLASS_CROPS[None, :] == crops[:, None]
    return (calibrated(name, mask) * keep).sum(axis=1)


sel_lg, sel_labels, sel_crops = part("selection")
OOF_R = cal.out_of_fold("temperature", within_crop(sel_lg, sel_crops), sel_labels, FOLDS, SEED)
FINAL_R = cal.fit("temperature", within_crop(sel_lg, sel_crops), sel_labels)
TEMPERATURE_R = FINAL_R.params["temperature"]


def rescaled(name, mask=None):
    """(confidence, confidence used for answering, correct) for the candidate."""
    lg, labels, crops = part(name, mask)
    probs = (OOF_R if mask is None else OOF_R[mask]) if name == "selection" else \
        FINAL_R.probabilities(within_crop(lg, crops))
    conf, ok = probs.max(axis=1), probs.argmax(axis=1) == labels
    return conf, np.where(crop_share(name, mask) >= 0.5, conf, 0.0), ok


conf_now, ok_now = shown(OOF, sel_labels, sel_crops)
conf_r, conf_r_answer, ok_r = rescaled("selection")
assert (ok_now == ok_r).all(), "rescaling changed a diagnosis"
ok = ok_now


def paired(statistic, a, b):
    """Bootstrap range of statistic(b) - statistic(a) over the same resampled photos."""
    return cal.bootstrap(lambda x, y, k: statistic(y, k) - statistic(x, k), a, b, ok, n=BOOTSTRAP, seed=SEED)


ece_now, ece_r = metrics.expected_calibration_error(conf_now, ok), metrics.expected_calibration_error(conf_r, ok)
aurc_now, aurc_r = cal.aurc(conf_now, ok), cal.aurc(conf_r_answer, ok)
ece_low, ece_high = paired(metrics.expected_calibration_error, conf_now, conf_r)
aurc_low, aurc_high = paired(cal.aurc, conf_now, conf_r_answer)
ADOPT = bool(ece_high < 0 and not aurc_low > 0)
A3 = {"temperature_within_crop": TEMPERATURE_R,
      "selection_out_of_fold": {"ece_now": ece_now, "ece_rescaled": ece_r, "ece_difference_range": [ece_low, ece_high],
                                "aurc_now": aurc_now, "aurc_rescaled": aurc_r, "aurc_difference_range": [aurc_low, aurc_high],
                                "mean_confidence_now": float(conf_now.mean()), "mean_confidence_rescaled": float(conf_r.mean()),
                                "accuracy": float(ok.mean())},
      "rule_a_ece_lower": bool(ece_high < 0), "rule_b_refers_no_worse": bool(not aurc_low > 0), "adopted": ADOPT}
print(json.dumps(A3, indent=1, default=float))
print("\nA3:", "ADOPTED - the threshold is re-derived below" if ADOPT else "NOT adopted - the deployed confidence stays")

In [ ]:
# The policy behind 0.8, as a cost: a wrong answer costs this many referrals.
COSTS = np.arange(1.0, 60.01, 0.5)
by_cost = pd.DataFrame({"cost of a wrong answer": COSTS,
                        "lowest-cost threshold now": [cal.threshold_for_cost(conf_now, ok, c, floor=FLOOR) for c in COSTS]})
by_cost["distance from 0.8"] = (by_cost["lowest-cost threshold now"] - DEPLOYED_THRESHOLD).abs()
COST = float(by_cost.sort_values(["distance from 0.8", "cost of a wrong answer"]).iloc[0]["cost of a wrong answer"])
print(f"0.8 matches a wrong answer costing {COST:g} referrals "
      f"(lowest-cost threshold at that cost: {cal.threshold_for_cost(conf_now, ok, COST, floor=FLOOR):.3f})")
if by_cost["distance from 0.8"].min() > 0.02 or COST in (COSTS[0], COSTS[-1]):
    print("WARNING: no cost in the range reproduces 0.8 closely; report this and do not re-derive a threshold from it.")
display(by_cost.iloc[::4].round(3))

NEW_THRESHOLD = cal.threshold_for_cost(conf_r_answer, ok, COST, floor=FLOOR) if ADOPT else DEPLOYED_THRESHOLD
A3["policy_cost_of_wrong_answer"] = COST
A3["threshold"] = NEW_THRESHOLD

a3_rows = []
for label, name, mask in SETS:
    _, labels, crops = part(name, mask)
    c_now, k = shown(calibrated(name, mask), labels, crops)
    c_r, c_r_answer, _ = rescaled(name, mask)
    for version, conf, answer, t in [("deployed", c_now, c_now, DEPLOYED_THRESHOLD),
                                     ("rescaled within crop", c_r, c_r_answer, NEW_THRESHOLD)]:
        at = cal.at_threshold(answer, k, t)
        a3_rows.append({"set": label, "version": version, "threshold": t, "accuracy": k.mean(),
                        "mean confidence": conf.mean(), "ECE": metrics.expected_calibration_error(conf, k),
                        "answered": at["coverage"], "accuracy of answered": at["selective_accuracy"],
                        "wrong per 100": 100 * at["wrong_and_answered"]})
a3 = pd.DataFrame(a3_rows).set_index(["set", "version"])
display(a3.round(4))

In [ ]:
import matplotlib.pyplot as plt

INK, MUTED, NOW, NEW = "#0b0b0b", "#52514e", "#1f6f43", "#b3261e"


def reliability_chart(file):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.6))
    for ax, (label, name) in zip(axes, [("Selection set (new field sources, out of fold)", "selection"),
                                         ("Test set (familiar sources)", "test")]):
        _, labels, crops = part(name)
        c_now, k = shown(calibrated(name), labels, crops)
        c_r, _, _ = rescaled(name)
        ax.plot([0, 1], [0, 1], color=MUTED, lw=1, ls="--")
        for conf, colour, text in [(c_now, NOW, "deployed"), (c_r, NEW, "rescaled within crop")]:
            r = cal.reliability(conf, k)
            ax.plot(r["confidence"], r["accuracy"], marker="o", ms=4, lw=2, color=colour,
                    label=f"{text} (ECE {metrics.expected_calibration_error(conf, k):.3f})")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_xlabel("Confidence shown", color=INK, fontsize=9)
        ax.set_ylabel("Share actually right", color=INK, fontsize=9)
        ax.set_title(label, color=INK, fontsize=10, loc="left")
        ax.legend(fontsize=8, frameon=False)
    plt.tight_layout()
    plt.savefig(OUT_DIR / file, dpi=150)
    plt.show()


reliability_chart("reliability.png")

## A4 - Per class at the deployed threshold (report only)

In [ ]:
def by_class(name, by, conf, ok, threshold):
    answered = conf >= threshold
    rows = PARTS[name]["rows"].assign(cls=lambda d: d["class"].map(SHORT), ok=ok, answered=answered,
                                      right_answered=answered & ok, wrong_answered=answered & ~ok)
    g = rows.groupby(by)
    out = pd.DataFrame({"photos": g.size(), "accuracy (crop given)": g["ok"].mean(), "answered": g["answered"].mean(),
                        "right and answered": g["right_answered"].sum(), "wrong and answered": g["wrong_answered"].sum()})
    out["accuracy of answered"] = out["right and answered"] / (out["right and answered"] + out["wrong and answered"])
    return out


a4 = {}
for title, name, by in [("selection by source and class", "selection", ["source", "cls"]),
                        ("test by class", "test", ["cls"]), ("PlantDoc by class", "ood_test", ["cls"])]:
    _, labels, crops = part(name)
    conf, k = shown(calibrated(name), labels, crops)
    a4[title] = by_class(name, by, conf, k, DEPLOYED_THRESHOLD)
    print(title, f"(threshold {DEPLOYED_THRESHOLD})")
    display(a4[title].round(3))

## A5 - Crop not given: detection, then the same rule (report only)

The backend picks the crop whose classes hold the most probability; if no crop holds at least
half, the crop is unclear and the photo is referred.

In [ ]:
a5_rows = []
for label, name, mask in SETS:
    _, labels, crops = part(name, mask)
    probs = calibrated(name, mask)
    totals = np.stack([(probs * (CLASS_CROPS == c)).sum(axis=1) for c in CROPS], axis=1)
    detected = np.array(CROPS)[totals.argmax(axis=1)]
    clear = totals.max(axis=1) >= 0.5
    conf, k = shown(probs, labels, detected)
    answered = (conf >= DEPLOYED_THRESHOLD) & clear
    a5_rows.append({"set": label, "photos": len(labels), "crop detected right": (detected == crops).mean(),
                    "crop unclear": 1 - clear.mean(), "accuracy, crop detected": k.mean(),
                    "answered at 0.8": answered.mean(),
                    "accuracy of answered": k[answered].mean() if answered.any() else np.nan,
                    "wrong per 100": 100 * (answered & ~k).mean()})
a5 = pd.DataFrame(a5_rows).set_index("set")
display(a5.round(3))

## Save the results

In [ ]:
a1.to_csv(OUT_DIR / "a1_calibration.csv")
a2.to_csv(OUT_DIR / "a2_thresholds.csv")
a3.to_csv(OUT_DIR / "a3_rescaling.csv")
by_cost.to_csv(OUT_DIR / "a3_policy_cost.csv", index=False)
for title, table in a4.items():
    table.to_csv(OUT_DIR / ("a4_" + title.replace(" ", "_") + ".csv"))
a5.to_csv(OUT_DIR / "a5_crop_not_given.csv")
training.save_json({"deployed": {"model_runs": MEMBERS, "temperature": TEMPERATURE, "threshold": DEPLOYED_THRESHOLD},
                    "A3": A3, "rules_written": "2026-10-31, before the first run"}, OUT_DIR / "decision.json")
if ADOPT:
    print("A3 adopted. Backend change (a separate PR): rescale the confidence within the crop,")
    print(f"  TEMPERATURE          = {TEMPERATURE_R:.4f} (fitted on the crop's classes only)")
    print(f"  CONFIDENCE_THRESHOLD = {NEW_THRESHOLD:.3f}")
    print("  and keep referring any photo whose crop holds under half the probability at T =", round(TEMPERATURE, 4))
else:
    print("A3 not adopted: nothing in the backend changes.")

## Copy the results into the repository, sync Drive, save to GitHub

Save the notebook first (Ctrl+S).

In [ ]:
for file in OUT_DIR.glob("*"):
    if file.suffix in {".csv", ".json", ".png"}:
        shutil.copy(file, REPO_RESULTS / file.name)
print(sorted(p.name for p in REPO_RESULTS.iterdir()))
drive.flush_and_unmount()
drive.mount("/content/drive")

In [ ]:
from google.colab import userdata

from src import github_sync

shutil.copy(ML_DIR / "notebooks" / "21_health_check.ipynb", REPO_RESULTS / "notebook.ipynb")
print(github_sync.push(
    REPO_DIR,
    branch="colab/health-check-a",
    message="Health check Part A: calibration, the threshold and the crop rule",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
    paths=[str(REPO_RESULTS.relative_to(REPO_DIR))],
))